# Phase 7: who gets hurt, and for how long

Sector employment growth is regressed on the unemployment change and lagged 10y changes (quarterly, from history), with
the sectors' residual covariance kept so sector shocks are drawn together. Recovery half-lives are measured on five US
episodes and on every simulated path (months from the unemployment peak until half the gap is closed).

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display, Markdown
import bond_sim.notebook as nb
pd.set_option("display.width", 170); pd.set_option("display.max_columns", 40); pd.set_option("display.max_rows", 80)
plt.rcParams["figure.dpi"] = 110

# ── parameters: change and rerun ─────────────────────────────────────────────
AS_OF = pd.Timestamp("2026-09-15")     # every loader filters on this date (point-in-time contract)
K = 1000                                 # Monte Carlo paths (raise for the paper)
FORCE_REFRESH = False                  # True re-downloads every series and rebuilds cached steps
os.environ["BOND_SIM_CACHE"] = "0" if FORCE_REFRESH else "1"
import bond_sim.config as bcfg
CFG_HASH = bcfg.load().content_hash()   # every cached step is keyed by the configuration that produced it
ctx = nb.cached(f"ctx_{AS_OF.date()}_{CFG_HASH}", lambda: nb.load_context(AS_OF), refresh=FORCE_REFRESH)
print(ctx.grid, "| config hash", CFG_HASH, "| series:", ctx.w.shape[1])

In [ ]:
from bond_sim.sim import LaborModel, RecoveryModel, ForcedTransition, FactorImpulse
book, auctions, agg, short = nb.cached(f"book_{AS_OF.date()}_{CFG_HASH}", lambda: nb.build_book(ctx), refresh=FORCE_REFRESH)
bf, chain = nb.cached(f"factors_chain_{AS_OF.date()}_{CFG_HASH}", lambda: nb.fit_factors_and_chain(ctx), refresh=FORCE_REFRESH)
var = nb.fit_var(ctx); init = nb.initial_state(ctx, book, auctions); H_, feas, h = nb.history_sustainability(ctx, agg)
labor = LaborModel(list(ctx.cfg.labor.sectors)).fit(ctx.w, ctx.qs)
display(labor.table().round(3))
rec = RecoveryModel()
display(rec.historical(ctx.w["UNRATE"].dropna())[["peak_date", "peak_u", "trough_u", "half_life_months"]])

In [ ]:
SHOCKS = {"none": None, "recession_now": [ForcedTransition(0, 0, 4)], "rate_spike": [FactorImpulse("rates", 2.0, 0, 0.8), FactorImpulse("financial", -1.5, 0, 0.7)],
          "consumer_retrenchment": [FactorImpulse("consumer", -2.0, 0, 0.8)]}
states_factory = lambda shocks: nb.make_blocks(ctx, var, bf, chain, shocks)["states"]
runs = nb.cached(f"runs_labor_{AS_OF.date()}_{CFG_HASH}_K1000", lambda: nb.run_scenarios(ctx, init, feas, states_factory, ["status_quo", "no_layoff_mandate", "austerity"], K, SHOCKS), refresh=FORCE_REFRESH)
rows, hl_rows = [], []
sector_paths = {}
for (s, p), r in runs.items():
    ok = r.admissible
    d_u = nb.quarterly_changes(r.paths["u"][ok], init.u_pct); d_r = nb.quarterly_changes(r.paths["r10"][ok], init.r10_pct)
    sector_paths[(s, p)] = (ok, labor.simulate(d_u, d_r, np.random.default_rng(ctx.cfg.sim.seed + 1)))
    hl = rec.simulated(r.paths["u"][ok], init.u_pct)
    hl_rows.append({"shock": s, "policy": p, "share_with_downturn": float(hl.notna().mean()), "median_half_life_months": float(hl.median()),
                    "p75_half_life_months": float(hl.quantile(0.75)), "median_peak_u": float(np.median(r.paths["u"][ok].max(axis=1)))})
# Sector damage is measured against the no-shock status-quo path on the same draws (common random numbers) at the same
# date, so secular trends (manufacturing, autos, federal employment all trend down over 30 years) cancel and what is
# left is the cyclical loss the shock or policy caused. Reported at its worst point within the first ten years.
ok0, base = sector_paths[("none", "status_quo")]
for (s, p), (ok, paths) in sector_paths.items():
    common = ok & ok0
    for sec_, path in paths.items():
        rel = 100.0 * (path[common[ok]] / base[sec_][common[ok0]] - 1.0)
        worst = rel[:, :40].min(axis=1)
        rows.append({"shock": s, "policy": p, "sector": sec_, "median_worst_gap_pct": float(np.median(worst)), "p05_worst_gap_pct": float(np.quantile(worst, 0.05))})
sec = pd.DataFrame(rows); hl_tab = pd.DataFrame(hl_rows)
print("Employment shortfall vs the no-shock status-quo path, worst point in the first 10 years (%), status quo by shock:")
display(sec[sec.policy == "status_quo"].pivot(index="sector", columns="shock", values="median_worst_gap_pct").round(1))
display(hl_tab.round(2))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 4.5))
sec[(sec.policy == "status_quo") & (sec["shock"] != "none")].pivot(index="sector", columns="shock", values="median_worst_gap_pct").plot.bar(ax=ax[0])
ax[0].set_title("Median employment shortfall vs no-shock path (%), status quo, by shock"); ax[0].legend(fontsize=7)
sec[(sec["shock"] == "recession_now")].pivot(index="sector", columns="policy", values="median_worst_gap_pct").plot.bar(ax=ax[1])
ax[1].set_title("Median employment shortfall vs no-shock status quo (%), recession-now shock, by policy"); ax[1].legend(fontsize=7)
plt.tight_layout(); plt.show()
fig, ax = plt.subplots(figsize=(11, 4))
for (s, p), r in runs.items():
    if p == "status_quo":
        hl = rec.simulated(r.paths["u"][r.admissible], init.u_pct).dropna()
        ax.hist(hl, bins=40, alpha=0.45, label=f"{s} (n={len(hl)})")
ax.set_title("Simulated recovery half-lives (months from unemployment peak), status quo, by shock"); ax.legend(fontsize=8); plt.show()

## What is still placeholder

The no-layoff mandate's floor and growth penalty (P-06), the austerity multiplier (P-07), the monetization inflation cost
(P-08), the growth uplift (P-09), the premium slope and form (P-01, P-02), and the premium-to-conditions mapping (P-16)
all still carry placeholder values. The sector betas are estimated but reduced-form (rates rise in booms, so the rate
coefficients partly reflect that); the unemployment channel carries the cyclical effect.